# Colab runner - full pipeline

Runs every stage of the assignment in order: data checks → SFT → reward model → PPO → DPO → evaluation.

**How to use**
1. *Runtime → Change runtime type →* a GPU (T4 works; A100 is much faster).
2. *Runtime → Run all.* If a stage fails, "Run all" stops there and the error is printed above.
3. If the runtime disconnects, reconnect and *Run all* again. Finished stages are skipped almost
   instantly, and an unfinished stage resumes from its last save (every 250 steps; PPO every 20 iterations).

Everything is stored on Google Drive in `MyDrive/rlhf/`: `checkpoints/`, `data_cache/`, `results/`.
Every stage prints `model dtype torch.float32` at start-up; if you see `bfloat16`, stop and pull the latest code.

In [ ]:
import os
import shutil
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

!pip install -q datasets rouge-score matplotlib

repo = Path("/content/Summarization-from-feedback")
if not (repo / "train_sft.py").is_file():
    !git clone https://github.com/Areesha-008/Summarization-from-feedback.git /content/Summarization-from-feedback
else:
    !git -C /content/Summarization-from-feedback pull
os.chdir(repo)
root = Path.cwd()

drive_root = Path("/content/drive/MyDrive/rlhf")
for name in ("checkpoints", "data_cache", "results"):
    target = drive_root / name
    target.mkdir(parents=True, exist_ok=True)
    link = root / name
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        for path in link.rglob("*"):
            if path.is_file():
                dest = target / path.relative_to(link)
                dest.parent.mkdir(parents=True, exist_ok=True)
                if not dest.exists():
                    shutil.copy2(path, dest)
        shutil.rmtree(link)
    link.symlink_to(target, target_is_directory=True)

print("project:", root)
print("data cache:", (root / "data_cache" / "data.pkl").exists())
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
!nvidia-smi -L

import subprocess


def run(cmd):
    """Run a script, stream its output, and stop "Run all" if it fails."""
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"`{cmd}` failed with exit code {proc.returncode}; see the output above.")


def show_md(path):
    from IPython.display import Markdown, display
    display(Markdown(open(path, encoding="utf-8").read()) if os.path.exists(path) else f"(no {path} yet)")


def show_img(path):
    from IPython.display import Image, display
    display(Image(path) if os.path.exists(path) else f"(no {path} yet)")

### Optional: retrain stages from scratch

Finished stages are reused. To retrain one, put its name in `RESET` and run this cell before the stages
(for example `RESET = ["dpo"]`). This deletes that stage's checkpoint and resume folders on Drive.

In [ ]:
RESET = []   # any of "sft", "reward", "ppo", "dpo"

for stage in RESET:
    for suffix in ("", "_resume", "_resume_tmp", "_resume_bak"):
        path = root / "checkpoints" / f"{stage}{suffix}"
        if path.exists():
            shutil.rmtree(path)
            print("deleted", path)

## Task 1 - Starter code, Table 1 and the SFT-mask check

In [ ]:
run("cd starter && python check_starter.py")
run("python common.py")
show_md("results/table1_data.md")

## Stage 1 - SFT (Task 2)

In [ ]:
run("python train_sft.py")

In [ ]:
show_md("results/sft_samples.md")
show_img("results/figures/fig1_sft_loss.png")

## Stage 2 - Reward model (Task 3)

Needs `checkpoints/sft/`.

In [ ]:
run("python train_reward.py")

In [ ]:
import pandas as pd
from IPython.display import display

display(pd.read_csv("results/logs/reward_log.csv"))
show_img("results/figures/fig2_reward.png")

## Stage 3 - PPO (Task 4)

Needs `checkpoints/sft/` and `checkpoints/reward/`. If the GPU runs out of memory, set `N_CHUNKS = 2` in `train_ppo.py`.

In [ ]:
run("python train_ppo.py")

In [ ]:
import pandas as pd
from IPython.display import display

display(pd.read_csv("results/logs/ppo_log.csv").tail(10))
show_img("results/figures/fig3_ppo.png")

## Stage 4 - DPO (Task 5)

Needs `checkpoints/sft/`.

In [ ]:
run("python train_dpo.py")

In [ ]:
import pandas as pd
from IPython.display import display

display(pd.read_csv("results/logs/dpo_log.csv"))
show_img("results/figures/fig4_dpo.png")

## Task 6 - Evaluation

Greedy summaries from SFT, PPO and DPO on the 300 evaluation posts, then Tables 2 and 3, the correctness
checks for the README, the blind human-evaluation sheet, and skeletons for `samples.md` and `analysis.md`.

In [ ]:
run("python evaluate.py")
for name in ("table3_final.md", "table2_training.md", "correctness_checks.md"):
    show_md(f"results/{name}")

## Download the results

Zips `results/` (logs, figures, tables, summaries, human-eval sheet) so you can copy it into the repo and commit it.
Checkpoints stay on Drive; do not commit them.

In [ ]:
import shutil as _sh
from google.colab import files

archive = _sh.make_archive("/content/results", "zip", root_dir=root, base_dir="results")
print("created", archive)
files.download(archive)

## After training: human evaluation (Section 3.2)

1. Open `results/human_eval.csv` (30 posts, summaries A/B/C in random order).
   Do **not** open `results/human_eval_key.csv` until both of you have finished.
2. Each member independently writes `A`, `B` or `C` in their own column (`member1_pick` / `member2_pick`).
3. Put the filled sheet back in `MyDrive/rlhf/results/` and run the cell below to build Table 4.

In [ ]:
run("python evaluate.py --tally-human")